In [3]:
import os 
from dotenv import load_dotenv
import logging
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS,InMemoryVectorStore
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger()
load_dotenv()

True

In [ ]:
class embedding:
    def __init__(self):
        self.embedding=OllamaEmbeddings(model='embeddinggemma')
        self.vectorstore=None
        self.index_path='faiss_index'
    def create_embed_vectorstore(self,chunks,batch_size=64):
        chunks=[c for c in (chunks or []) if c.page_content.strip()]
        if not chunks:
            raise ValueError('non empty chunks available to embed')
        else:
            try:
                self.vectorstore=FAISS.from_documents(chunks[:batch_size],self.embedding)
                for i in range(batch_size,len(chunks),batch_size):
                    self.vectorstore.add_documents(chunks[i:i+batch_size])
                    logger.info("embedd %d/%d chunks", min(i+batch_size,len(chunks)),len(chunks))
                logger.info('vectorstore built with %d vectors',self.vectorstore.index.ntotal)
                return self.vectorstore
            except Exception:
                logger.exception("vectorstore embedding failed ")
                raise
    def save(self,path=None):
        if self.vectorstore is None:
            raise RuntimeError('no vectorstore to save, call and create vector store first')
        self.vectorstore.save_local(path or self.index_path)
        logger.info('index saved to ',path or self.index_path)

    def load(self,path=None):
        if self.vectorstore is None:
            raise RuntimeError('vectorstore not ready , call and create a vectorstore')
        self.vectorstore=FAISS.load_local(path,self.embedding, allow_dangerous_deserialization=True)
    def search(self,query,k:int):
        if self.vetorstore is None :
            raise RuntimeError('vector store is empy create a vectorstore')
        return self.vectorstore.similarity_search(query,k=k,)
    def retriver(self, k:int):
        if self.vectorstore is None:
            raise RuntimeError('vector store is empty or not intialized call and create a vectorstore')
        return self.vectorstore.as_retriever(search_kwargs={k:k})
     




